# **Tools**

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.retrievers import ArxivRetriever
from langchain_community.retrievers import WikipediaRetriever
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
from langchain.tools import tool

if os.environ['GROQ_API_KEY']:
    print("GROQ_API_KEY is set.")
else:
    raise ValueError("GROQ_API_KEY is not set. Please set it in your .env file.")

GROQ_API_KEY is set.


In [2]:
llm = ChatGroq(model='meta-llama/llama-4-scout-17b-16e-instruct', temperature=0.5)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D462A346E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D462A35160>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', temperature=0.5, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

'1 day ago - Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Obama previously served as a U.S. senator representing Illinois from ... 2 days ago - Barack Hussein Obama II (/bəˈrɑːk huːˈseɪn oʊˈbɑːmə/ ; born August 4, 1961) is an American politician and attorney. He was the 44th president of the United States from 2009 to 2017. November 25, 2025 - Obama is a surname. It most commonly refers to Barack Obama (born 1961), the 44th president of the United States. Obama is a common Fang masculine name in western Central Africa. 1 month ago - Barack Hussein Obama (/ˈbærək huːˈseɪn oʊˈbɑːmə/; born Baraka Hussein Obama, 18 June 1934 – 24 November 1982) was a Kenyan senior governmental economist and the father of Barack Obama, the 44th president of the United States. 3 weeks ago - The selection was slow because Malia is al

In [4]:

search = DuckDuckGoSearchResults()

search.invoke("Obama")

'snippet: Barack Hussein Obama II was born on August 4, 1961 [3] in Kapiʻolani Medical Center for Women and Children (called Kapiʻolani Maternity & Gynecological Hospital in 1961) in Honolulu, Hawaii. [4][5] He is the first President to have been born in Hawaii. [6] His father was a black exchange student from Kenya named Barack Obama Sr. He died in a motorcycle accident in Kenya in 1982. His mother ..., title: Barack Obama - Simple English Wikipedia, the free encyclopedia, link: https://simple.wikipedia.org/wiki/Barack_Obama, snippet: Official portrait of Barack Obama Barack Hussein Obama II (born August 4, 1961) is an American politician and attorney who served as the 44th president of the United States from January..., title: Barack Obama, link: https://grokipedia.com/page/Barack_Obama, snippet: Barack Obama, the 44th President of the United States, broke barriers as the first African-American president and implemented significant healthcare reforms during his tenure., title: Barack

In [7]:

arxiv_retriever = ArxivRetriever(
    load_max_docs=2,
    get_full_documents=True,
)

In [ ]:
docs = arxiv_retriever.invoke("Attention is all you need")
docs[0].metadata

## **Another Tool**

In [3]:
wiki_retriever = WikipediaRetriever()

In [4]:
docs = wiki_retriever.invoke("TOKYO GHOUL")
docs

[Document(metadata={'title': 'Tokyo Ghoul', 'summary': "Tokyo Ghoul (Japanese: 東京喰種（トーキョーグール）, Hepburn: Tōkyō Gūru) is a Japanese dark fantasy manga series written and illustrated by Sui Ishida. It was serialized in Shueisha's seinen manga magazine Weekly Young Jump from September 2011 to September 2014, with its chapters collected in 14 tankōbon volumes. The manga has been licensed for English release in North America by Viz Media.\nThe story is set in an alternate version of Tokyo where humans coexist with ghouls, beings who look like humans but can only survive by eating human flesh. Ken Kaneki is a college student who is transformed into a half-ghoul after an encounter with one of them. He must navigate the complex social and political dynamics between humans and ghouls while struggling to maintain his humanity.\nA prequel, titled Tokyo Ghoul [Jack], ran online on Jump Live in 2013, with its chapters collected in a single tankōbon volume. A sequel, titled Tokyo Ghoul:re, was serial

## **Arxiv Query Run**

In [5]:
arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
result = arxiv_query.invoke("Attention is all you need")
result

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifically, we replace the attention layer in a vision transformer with a feed-forward layer applied over the patch dimension. The resulting architecture is simply a series of feed-forward layers applied over the patch and feature dimensions in an alternating fashion. In experiments on ImageNet, this architecture performs surprisingly well: a ViT/DeiT-base-sized model obtains 74.9\\% top-1 accuracy, compared to 77.9\\% and 79.9\\% for ViT and DeiT respectively. These results indicate that aspects

## **Custom Tools**

In [6]:
@tool('get_info')
def get_info(name: str)-> str:
    """
    Takes a String Argument and returns information about a person. If the person is not found, it returns "Info not found."
    """
    
    info = {
        "Tapabrata": "Tapabrata is a software engineer with expertise in machine learning and natural language processing.",
        "Alice": "Alice is a data scientist who specializes in deep learning and computer vision.",
        "Bob": "Bob is a researcher in the field of artificial intelligence, focusing on reinforcement learning and robotics."
    }
    return info.get(name, "Info not found.")

get_info.invoke("Tapabrata")

'Tapabrata is a software engineer with expertise in machine learning and natural language processing.'

## **Tool Binding**

In [7]:
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.tools import tool
from langchain.agents import create_agent
from pprint import pprint
if os.getenv('HUGGINGFACEHUB_API_TOKEN'):
    print("HUGGINGFACEHUB_API_TOKEN is set.")
else:
    raise ValueError("HUGGINGFACEHUB_API_TOKEN is not set. Please set it in your .env file.")

# 1. Define your custom tool with precise type hints
@tool
def search_duckduckgo(query: str) -> str:
    """Use this tool to search DuckDuckGo for live, real-time information or recent events."""
    search_ddg = DuckDuckGoSearchResults()
    return str(search_ddg.invoke(query))

# 2. Configure the LLM
hf_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct", 
    task="text-generation"
)

# 3. Use the chat model interface wrapper
chat_model = ChatHuggingFace(llm=hf_endpoint)
tools = [search_duckduckgo]

# 4. Create the modern agent runner 
# (This replaces the deprecated langgraph factory)
agent = create_agent(chat_model, tools=tools)

# 5. Invoke the updated agent executor structure
inputs = {"messages": [("user", "Give me Narendra Modis Todays Visit to Italy latest News")]}
result = agent.invoke(inputs)

# 6. Output the finalized reasoning answer 
print("\nFinal Result:")
pprint(result["messages"][-1].content)


HUGGINGFACEHUB_API_TOKEN is set.


d:\GEN-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Final Result:
("The latest news on Narendra Modi's visit to Italy is available on various "
 'news websites. During his visit, PM Modi received a warm welcome from the '
 'Indian diaspora and met with Italian PM Giorgia Meloni to discuss boosting '
 'India-Italy cooperation. The two leaders are expected to work towards '
 'upgrading bilateral ties into a special strategic partnership. You can find '
 'more updates and details on news websites such as Financial Express, '
 'Hindustan Times, and News18.')
